# Kashmiri word OCR: characters vs. PAW units (v2)

CNN + BiLSTM + CTC word recognizer for printed Kashmiri, trained on:

| Data | Images | Notes |
|---|---|---|
| Kaggle **nawabhussaen/kashmiri-ocr** | 69,434 | mostly Naskh-style fonts; licence *unknown* (research use) |
| **Synthetic**, drawn here | 120,000 | Nastaliq (Noto Nastaliq Urdu, Gulzar) and Naskh fonts; words sampled half by real-text frequency (Bolbosh + Kashmiri-Language-Corpus counts in `ks-lexicon.json`) and half across the 106,000-form lexicon |
| HF **Omarrran/40K_kashmiri_text_and_image_dataset** | 40,799 | optional: needs a Kaggle secret `HF_TOKEN` and accepted terms (non-commercial use) |
| HF **Omarrran/600k_KS_OCR_Word_Segmented_Dataset** | ~602,000 | optional: needs `HF_TOKEN` and the owner's manual approval |

Two runs on the same data and split: **character** units, and **PAW** units (every piece of a word between non-joining letters seen at least 25 times, e.g. کٲشُر = کٲ + شُر; rarer pieces fall back to characters). Train/val/test are split by the word text, so test words are never seen in training.

**Before running:** Accelerator **GPU**, **Internet on**, *Add Input* → **kashmiri-ocr** (by nawabhussaen). Optional: *Add-ons → Secrets* → `HF_TOKEN`.

In [ ]:
!git clone --depth 1 https://github.com/Rizvonium/kashmiri_ocr.git /kaggle/working/kashmiri_ocr
%cd /kaggle/working/kashmiri_ocr
!pip install -q fonttools onnx huggingface_hub
import torch, glob, os, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the accelerator to GPU")
hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/labels", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(p), "word_images"))]
assert hits, "Attach the 'kashmiri-ocr' dataset by nawabhussaen (Add Input)"
DATA = [hits[0]]; print("Kaggle dataset:", DATA[0])

In [ ]:
# Open-licence fonts (SIL OFL) from the Google Fonts repository
FONTS = {"NotoNastaliqUrdu.ttf": "ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf",
         "Gulzar-Regular.ttf": "ofl/gulzar/Gulzar-Regular.ttf",
         "NotoNaskhArabic.ttf": "ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf",
         "ScheherazadeNew-Regular.ttf": "ofl/scheherazadenew/ScheherazadeNew-Regular.ttf"}
os.makedirs("fonts", exist_ok=True)
for name, path in FONTS.items():
    subprocess.run(["curl", "-sSL", "-o", f"fonts/{name}", f"https://cdn.jsdelivr.net/gh/google/fonts@main/{path}"], check=True)
from PIL import features
if not features.check("raqm"):
    subprocess.run("apt-get -qq install -y libraqm0 libfribidi0", shell=True)
    print("installed libraqm - if the next line says False, Run > Restart and run from the top")
import importlib, PIL.features; importlib.reload(PIL.features); print("raqm (Arabic shaping):", PIL.features.check("raqm"))

In [ ]:
# Optional gated Hugging Face sets (skipped without a Kaggle secret HF_TOKEN or without access)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    from huggingface_hub import snapshot_download
    for repo, out in [("Omarrran/40K_kashmiri_text_and_image_dataset", "/kaggle/tmp/hf40k"),
                      ("Omarrran/600k_KS_OCR_Word_Segmented_Dataset", "/kaggle/tmp/hf600k")]:
        try:
            local = snapshot_download(repo, repo_type="dataset", token=HF_TOKEN, allow_patterns=["*.zip"],
                                      local_dir=f"/kaggle/tmp/{repo.split('/')[1]}")
            for z in sorted(glob.glob(f"{local}/**/*.zip", recursive=True)):
                subprocess.run(["python", "training/prepare_external.py", z, out, "--name", repo.split("/")[1]], check=True)
                os.remove(z)  # free disk as we go
            DATA.append(out); print("added", repo)
        except Exception as e:
            print(f"skipped {repo}: {type(e).__name__}: {str(e)[:200]}")
else:
    print("no HF_TOKEN secret: training without the Hugging Face image sets")
print("data:", DATA)

In [ ]:
# 120,000 synthetic word images, 70% Nastaliq (about 8 minutes)
!python training/synth_words.py --fonts fonts --out /kaggle/tmp/synth --n 120000 --nastaliq-share 0.7 --freq-share 0.5
DATA.append("/kaggle/tmp/synth")

In [ ]:
# Run 1: character units
!python training/train_word_ocr.py --data {" ".join(DATA)} --units chars --epochs 20 --batch 128 --workers 4 --out /kaggle/working/runs/chars

In [ ]:
# Run 2: PAW units (the expanded space), same data and split
!python training/train_word_ocr.py --data {" ".join(DATA)} --units paws --epochs 20 --batch 128 --workers 4 --out /kaggle/working/runs/paws

In [ ]:
# Compare on unseen test words
import json, pandas as pd
rows = []
for run in ("chars", "paws"):
    r = json.load(open(f"/kaggle/working/runs/{run}/report.json"))
    for group, v in r["test"].items():
        rows.append({"units": f"{run} ({r['n_outputs']})", "group": group, "n": v["n"], "CER": round(v["CER"], 4),
                     "WER": round(v["WER"], 4), "WER + dictionary": round(v["WER_with_dictionary"], 4)})
pd.DataFrame(rows).pivot_table(index="group", columns="units", values=["CER", "WER", "WER + dictionary"])

**Outputs** in `/kaggle/working/runs/<units>/`: `best.pt`, `kashmiri_word_ocr.onnx` + `.json` (for the web app via onnxruntime-web) and `report.json`. Synthetic and converted images are deleted at the end to keep the saved output small.

In [ ]:
import shutil
for d in ("/kaggle/tmp/synth", "/kaggle/tmp/hf40k", "/kaggle/tmp/hf600k", "/kaggle/working/kashmiri_ocr"):
    shutil.rmtree(d, ignore_errors=True)
!ls -la /kaggle/working/runs/*